# Custom LoRA via Multi-GPU Training

In [ ]:
# Check for multi-GPU access on the system..
! nvidia-smi

In [ ]:
import warnings
warnings.filterwarnings("ignore")
from datasets import load_dataset

## Data Preprocessing
We would use the openassistant guanaco dataset for training.

In [ ]:
ds = load_dataset("timdettmers/openassistant-guanaco")

Let us view 5 samples from the dataset

In [ ]:
for sample in ds['train'].select(range(5)):
    print(f"\n {'*' * 64}\n{sample}\n{'*' * 64}")

We can see dict objects with `text` key in multiple languages. For this lab, we limit ourselves to the english samples

In [ ]:
from langdetect import detect

def remove_nonEnglish_rows(ds):
    # Initialize an empty list to store rows detected as English
    new_ds = []
    
    # Initialize a list to store indices of rows that cause issues (corner cases)
    corner_case = []
    
    # Iterate through each row in the dataset's 'text' column
    for i, row in enumerate(ds['text']):
        try:
            # Detect the language of the text
            if detect(str(row)) == 'en':
                # If the language is English, add the row to new_ds
                new_ds.append(row)
        except:
            # If an exception occurs, add the index to corner_case
            corner_case.append(i)
    
    # Return the list of English rows and the indices of corner cases
    return new_ds, corner_case


In [ ]:
filter_train_samples,cc_train = remove_nonEnglish_rows(ds['train'])

print("Count of training samples: ",len(filter_train_samples))

In [ ]:
filter_test_samples,cc_test = remove_nonEnglish_rows(ds['test'])
print("Count of testing samples: ",len(filter_test_samples))


In [ ]:
# save English text samples
import json
def save_jsonl(ds,filename):
    with open(f"data/{filename}.jsonl", "w") as write_file:
            json.dump(ds, write_file, indent=4)
            print("dataset saved in jsonl format ....")

The LLaMA3 models expect the train datasets to be in a specific format as listed [here](https://llama.meta.com/docs/model-cards-and-prompt-formats/meta-llama-3/)

In [ ]:
def transform_to_template(example):
    conversation_text = example['text']
    segments = conversation_text.split("###")[1:]
    

    for idx,segment in enumerate(segments):
        if idx%2==0:
            segments[idx] = segment.replace('Human:',"<|start_header_id|>user<|end_header_id|>") + "<|eot_id|>"
        else:
            segments[idx] = segment.replace('Assistant:',"<|start_header_id|>assistant<|end_header_id|>") + "<|eot_id|>"
    
    

    segments = ["<|begin_of_text|><|start_header_id|>system<|end_header_id|> You are a helpful assistant<|eot_id|>"] + segments

    return {'text': ''.join(segments)}

We will make a seperate folder for storing these datasets.

In [ ]:
! mkdir -p data
! mkdir -p data/filtered

In [ ]:
# set file names  
save_train_filename = 'filtered/train'
save_test_filename = 'filtered/test'

# save file
save_jsonl(filter_train_samples, save_train_filename)
save_jsonl(filter_test_samples, save_test_filename)

In [ ]:
dataset = load_dataset('data/filtered/')

In [ ]:
template_dataset = dataset.map(transform_to_template)

In [ ]:
!mkdir -p data/ds_preprocess
template_dataset.save_to_disk('data/ds_preprocess/')

We have now saved the transformed dataset. This would help in loading the dataset directly in case of kernel failures.

## Multi-GPU Training Setup

### Code Execution Strategy

In this notebook, we are not executing the multi-GPU training code directly. Instead, we use the `%%writefile` magic command to write the training script into a standalone Python file. This approach ensures that the script can be executed later using `torchrun`, which is specifically designed for distributed training across multiple GPUs.

`torchrun` handles multi-GPU execution efficiently by managing process spawning and inter-GPU communication. Running the script separately also helps avoid potential conflicts with Jupyter's interactive execution model, making it more suitable for large-scale distributed training.


In [ ]:
%%writefile multi_gpu_lora.py

import os
import torch
import torch.distributed as dist
import datetime
from peft import LoraConfig, PeftModel
from trl import SFTTrainer,SFTConfig
from datasets import load_from_disk
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)

In [ ]:
%%writefile -a multi_gpu_lora.py

# Base model and data paths
base_model = "/mnt/lustre/docker-strg/llama3_8b_weights"
data_path = "data/ds_preprocess/train"
eval_path = "data/ds_preprocess/test"

### Distributed Training Setup in PyTorch

In this section, we configure our environment for distributed training using PyTorch's `torch.distributed` package. Distributed training allows us to leverage multiple GPUs to accelerate model training and handle larger datasets efficiently.

#### Key Concepts and Technologies

- **[Distributed Training](https://pytorch.org/tutorials//distributed/home.html)**: This approach involves spreading the training workload across multiple GPUs or machines, significantly improving training speed and model accuracy. It's particularly beneficial for large models and compute-intensive tasks.  

- **`torch.distributed` Package**: PyTorch provides this [package](https://pytorch.org/docs/stable/distributed.html) to support distributed training. It offers communication primitives for multiprocess parallelism across several computation nodes running on one or more machines.  

- **NCCL Backend**: The NVIDIA Collective Communication Library (NCCL) implements multi-GPU and multi-node communication primitives optimized for NVIDIA GPUs and Networking. NCCL provides routines such as all-gather, all-reduce, broadcast, reduce, reduce-scatter as well as point-to-point send and receive that are optimized to achieve high bandwidth and low latency over PCIe and NVLink high-speed interconnects within and across nodes. It is the recommended backend for GPU-based distributed training in PyTorch. [More details can be found here](https://developer.nvidia.com/nccl)

- **Environment Variables**:
  - `LOCAL_RANK`: Indicates the rank of the process on the local machine. Each process is assigned a unique local rank, starting from 0 up to the number of GPUs per node minus one.
  - `WORLD_SIZE`: Specifies the total number of processes participating in the job across all nodes.

#### Process Initialization

The `setup_distributed` function initializes the distributed environment:

1. **Retrieve Environment Variables**: Fetches `LOCAL_RANK` and `WORLD_SIZE` to determine the process's role and the total number of processes.

2. **Set Device**: Assigns the current process to the GPU corresponding to its local rank using `torch.cuda.set_device(local_rank)`.

3. **Initialize Process Group**: Establishes communication between processes by initializing the process group with the NCCL backend:

   ```python
   dist.init_process_group(
       backend="nccl",
       init_method="env://",
       world_size=world_size,
       rank=local_rank
   )
   ```

In [ ]:
%%writefile -a multi_gpu_lora.py

import os
import torch
import torch.distributed as dist

def setup_distributed():
    """ Initialize distributed training properly by setting up the process group and device assignment. """
    
    # Get the local rank of the current process (assigned by torchrun)
    local_rank = int(os.environ.get("LOCAL_RANK", -1))
    
    # Get the total number of processes involved in training
    world_size = int(os.environ.get("WORLD_SIZE", 1))
    
    if local_rank != -1:  # Ensure distributed training is enabled
        # Set the CUDA device for the current process
        torch.cuda.set_device(local_rank)
        
        # Initialize the process group for multi-GPU communication
        dist.init_process_group(
            backend="nccl",      # Use NCCL backend for efficient GPU communication
            init_method="env://", # Use environment variables for initialization
            world_size=world_size, # Total number of processes
            rank=local_rank        # Rank of the current process
        )
    
    return local_rank  # Return local rank for further use

# Initialize distributed training
local_rank = setup_distributed()

In [ ]:
%%writefile -a multi_gpu_lora.py

# Load datasets from disk
dataset = load_from_disk(data_path)       # Load the training dataset
eval_dataset = load_from_disk(eval_path)  # Load the evaluation dataset

# Initialize the tokenizer using the base model
tokenizer = AutoTokenizer.from_pretrained(base_model)

# Ensure padding is handled correctly
tokenizer.pad_token = tokenizer.eos_token  # Set pad token to EOS for compatibility
tokenizer.padding_side = "right"           # Right-padding ensures consistency for batch processing

In [ ]:
%%writefile -a multi_gpu_lora.py

# Quantization configuration for memory-efficient training
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,                      # Enable 4-bit quantization for reduced memory footprint
    bnb_4bit_quant_type="nf4",              # Use NormalFloat4 (NF4) quantization format
    bnb_4bit_compute_dtype=torch.float16,   # Perform computations in 16-bit precision
    bnb_4bit_use_double_quant=True,         # Use double quantization for better accuracy
)

# Training configuration for fine-tuning
training_config = SFTConfig(
    output_dir="model/training_results",    # Directory to save model checkpoints
    per_device_train_batch_size=16,          # Batch size per GPU for training
    per_device_eval_batch_size=8,           # Batch size per GPU for evaluation
    gradient_accumulation_steps=2,          # Accumulate gradients to simulate larger batch sizes
    learning_rate=2e-4,                      # Initial learning rate
    weight_decay=0.001,                      # Weight decay for regularization
    num_train_epochs=2,                      # Number of training epochs
    lr_scheduler_type="cosine",              # Use cosine learning rate schedule
    warmup_ratio=0.03,                       # Warmup ratio for learning rate scheduler
    log_level="warning",                      # Set logging level to reduce verbosity
    logging_steps=25,                         # Log training progress every 25 steps
    save_steps=25,                            # Save model checkpoints every 25 steps
    fp16=True,                                # Enable mixed-precision training for efficiency
    local_rank=local_rank,                    # Assign rank for distributed training
    ddp_backend="nccl",                       # Use NCCL backend for optimized GPU communication
    optim="adamw_torch_fused",                # Use fused AdamW optimizer for better performance
    gradient_checkpointing=True,              # Enable gradient checkpointing to save memory
    ddp_find_unused_parameters=False,         # Optimize DDP by avoiding unused parameter checks
    dataset_text_field="text",                # Define the dataset text field for tokenization
)

# LoRA (Low-Rank Adaptation) configuration for efficient fine-tuning
peft_params = LoraConfig(
    lora_alpha=16,                # Scaling factor for LoRA adaptation
    lora_dropout=0.1,             # Dropout rate for LoRA layers
    r=64,                         # Rank for the low-rank decomposition
    bias="none",                   # Disable bias adjustment
    task_type="CAUSAL_LM",         # Task type: Causal Language Modeling
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],  # Target transformer layers for LoRA
)

In [ ]:
%%writefile -a multi_gpu_lora.py

if __name__ == "__main__":
    try:
        # Load the base model with LoRA and quantization settings
        model = AutoModelForCausalLM.from_pretrained(
            base_model,
            torch_dtype=torch.float16,     # Use 16-bit precision for efficiency
            quantization_config=quant_config,  # Apply 4-bit quantization settings
        )
        model.config.use_cache = False  # Disable caching to support gradient checkpointing

        # Initialize the fine-tuning trainer
        trainer = SFTTrainer(
            model=model,
            tokenizer=tokenizer,  # Tokenizer for preprocessing
            train_dataset=dataset.select(range(1000)), 
            eval_dataset=eval_dataset,  # Evaluation dataset
            peft_config=peft_params,  # LoRA configuration
            args=training_config,  # Training hyperparameters
        )

        # Start fine-tuning
        trainer.train()

        # Save the fine-tuned model and tokenizer
        new_model = "model/Llama-3-8b-instruct-hf-finetune-multigpu"
        trainer.model.save_pretrained(new_model)
        trainer.tokenizer.save_pretrained(new_model)
        trainer.model.save_pretrained(new_model,safe_serialization=False)

        print("Training Complete..")
    
    except Exception as e:
        print(e)  # Print any errors that occur
        pass  # Continue execution without crashing

    # Clean up the distributed training process
    dist.destroy_process_group()

### Running the Multi-GPU Training Script

Now that we have written the multi-GPU training script (`multi_gpu_lora.py`), we need to execute it using `torchrun`, the recommended launcher for distributed training in PyTorch. This ensures that multiple GPU processes are initialized properly and communicate efficiently.

#### Steps to Run the Training Script

1. **Open a New Terminal in JupyterLab and Activate Environment**  
   - In JupyterLab, click on **File** → **New Launcher** → **Terminal**  
   - This opens a new shell terminal inside your Jupyter environment.
   - Run `conda deactivate` command to exit the base environment and enable the active python environment `/mnt/lustre/python_env/nims_env`.

2. **Navigate to the Script Directory**  
   - If needed, change into the directory where `multi_gpu_lora.py` is saved:
     ```bash
     cd /path/to/your/script/
     ```

3. **Run the Training Script with `torchrun`**  
   The following command will automatically detect the number of GPUs available and launch the script accordingly:
   ```bash
   torchrun --standalone --nnodes=1 --nproc_per_node=$(nvidia-smi -L | wc -l) multi_gpu_lora.py

    Explanation of the Command:
    	•	torchrun: PyTorch’s distributed training launcher.
    	•	--standalone: Runs on a single node without requiring an external rendezvous server.
    	•	--nnodes=1: Specifies that we are running on a single machine (node).
    	•	--nproc_per_node=$(nvidia-smi -L | wc -l):
    	•	Automatically detects the number of available GPUs using nvidia-smi.
    	•	The command nvidia-smi -L | wc -l counts the number of GPUs and sets that value dynamically.
    	•	multi_gpu_lora.py: The training script we wrote.

4.	Monitor Training Output
	•	Once executed, you should see logs indicating that multiple processes have been initialized and training has begun.
	•	Ensure that logs from different ranks (processes) are being printed, confirming multi-GPU execution.

Verifying GPU Utilization

To confirm that all GPUs are being used, you can run:

`nvidia-smi`

This will show GPU memory usage and active processes.

This setup ensures that your training script dynamically scales based on the number of GPUs available, making it portable across different hardware setups.

---

## Licensing

Copyright © 2025 OpenACC-Standard.org. This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). These materials include references to hardware and software developed by other entities; all applicable licensing and copyrights apply.